# BNB MC+Overlay Truth Analysis -- ICARUSNuGraphAnalyzer

## Run command

```
lar -c nugraph_analyser_wtruth_v2.fcl -S <files_list>.list -n -1
```

where `<files_list>.list` is a subset of
`/pnfs/icarus/scratch/users/sdey2/v10_06_00_04p04/nugraph/BNB_NuGraph_v10_06_00_04p04_test/stage1/5000/files.list`
(BNB MC + cosmic overlay, WireCell/2D-detsim chain -- `stage0_run2_wcdnn_icarus_overlay_...` stage1 files).

## Output

Tree `NuGraphAnaCryoE/NuGraphHitOutput` (+ `NuGraphEventOutput`,
`NuGraphEdepOutput`, `NuGraphSliceOutput`, `NuGraphPFPOutput`).

## What was different about getting truth to work on THIS sample type

The truth-inclusive analyzer worked cleanly out of the box on the MPVMPR
pure-MC training sample (97.7% truth-match rate). Pointed at this BNB
overlay sample with no changes, it silently returned **0% truth match on
every hit** -- no crash, no error, just `truth_semantic == -1` everywhere.

Investigated and ruled out, each checked directly against this file's
actual contents rather than assumed from fcl config:
- `SimChannelModuleLabel: "merge"` -- confirmed present and correctly
  labeled in the file (`sim::SimChannels_merge__DetSim`). Not the cause.
- `ParticleInventoryService.OverrideRealData` (`true` vs `false`) -- bit
  identical result either way. Not the cause.
- `largeant` as `G4ModuleLabel` -- confirmed present and correct
  (`simb::MCParticles_largeant__G4`). Not the cause.

**Actual cause**: live `BackTrackerService::ChannelToTrackIDEs` does not
work correctly for this overlay production's timing/channel-mixing
mechanism. The production already ships a precomputed workaround: an
`art::Assns<recob::Hit, simb::MCParticle, anab::BackTrackerHitMatchingData>`
under the label `mcassociationsGausCryoE`.

**Fix**: added a second truth-matching code path to the analyzer
(`GetContributors()` + new fcl param `HitTruthAssnsLabel`). When
`HitTruthAssnsLabel: "mcassociationsGausCryoE"` is set, the analyzer reads
this precomputed Assns instead of calling live backtracking. Confirmed
working: truth-match rate went from 0% to ~11-12%, with 100% of matched
hits correctly tracing to the neutrino interaction (`truth_isnu==1`) --
exactly the expected shape for an overlay sample, where most hits are real
cosmic data with no simulation truth at all.

**Open question, not yet resolved**: why did the same general fcl approach
work for other MC+overlay samples before? Best guess: the existing
production CAFMaker chain may never have relied on live
`ChannelToTrackIDEs` for this in the first place -- it may already read
this same Assns product internally. Worth checking `sbncode/CAFMaker`
directly to confirm.

In [ ]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmcrameri.cm as cmc
from sklearn.metrics import confusion_matrix
from concurrent.futures import ThreadPoolExecutor, as_completed

%matplotlib inline

# --- style: serif font, no gridlines, Managua colourscheme ---
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["axes.grid"] = False
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

MANAGUA = cmc.managua
MANAGUA_R = cmc.managua_r
def managua_discrete(n, reverse=False):
    cmap = MANAGUA_R if reverse else MANAGUA
    return [cmap(x) for x in np.linspace(0.08, 0.92, n)]

score_cols = ["MIP", "HIP", "shower", "michel", "diffuse"]
class_labels = ["MIP", "HIP", "Shower", "Michel", "Diffuse"]

## Stream directly from the flat CAF files

The two 20 GB pickle files are deliberately **not used** below. A pickle cannot
be read in chunks, and loading even one can require far more than 20 GB of RAM.

Instead, this notebook reads one flat CAF at a time from `files_manual_filter.list`,
updates compact running aggregates, releases that file, and continues. A small
checkpoint is written every 50 files, so the job can resume after a disconnected
Jupyter page or kernel restart without starting again.

The existing `batch_0.pkl` and `batch_1.pkl` can remain on disk, but this notebook never opens them.

### Recommended workflow

1. Leave `MAX_FILES = 200` and run through the entire notebook to verify every
   table, plot, and bootstrap step.
2. When the analysis looks correct, change only `MAX_FILES = None`.
3. Rerun the streaming cell. It will resume at file 200 using the same compact
   checkpoint and continue through the remaining CAFs.
4. Rerun the aggregate-loading and plotting cells for the final figures.

Do not set `START_FRESH = True` when switching from test mode to the full run,
because doing so would discard the first 200 processed files.


In [ ]:
file_list_path = "/exp/icarus/data/users/sdey2/cafanalysisbatch_Aug3/files_flatcafs.list"

tree_path = "NuGraphAnaCryoE/NuGraphHitOutput"
load_cols = [
    "event",
    "truth_semantic",
    "truth_isnu",
    "truth_g4id",
    "has_ng_output",
    "x_filter",
    *score_cols,
]

# Small restartable aggregate checkpoint; this is NOT a hit-level dataframe.
AGGREGATE_CHECKPOINT = "/exp/icarus/data/users/sdey2/cafanalysisbatch_Aug3/direct_streaming_aggregates.npz"
CHECKPOINT_EVERY = 50       # save after this many CAFs
MAX_FILES = None             # test mode; set to None for all ~5000 files
START_FRESH = False         # set True once to discard an old aggregate checkpoint
N_CLASSES = len(score_cols)
X_FILTER_BINS = np.linspace(0.0, 1.0, 41)

import os
import gc
import time

with open(file_list_path) as fh:
    full_path_list = [line.strip() for line in fh if line.strip()]

all_paths = (
    full_path_list
    if MAX_FILES is None
    else full_path_list[:MAX_FILES]
)

mode = "FULL RUN" if MAX_FILES is None else f"TEST MODE ({MAX_FILES} files)"
print(f"{len(full_path_list):,} flat CAF paths found in total")
print(f"{mode}: {len(all_paths):,} files selected")
print(f"aggregate checkpoint: {AGGREGATE_CHECKPOINT}")

In [ ]:
# Run this cell once. In test mode it stops after MAX_FILES and writes a
# valid compact checkpoint for all downstream plots. Later, set MAX_FILES=None
# and rerun: it resumes from the next unprocessed file rather than starting over.
#
# It streams over the selected CAF list and can resume from its compact checkpoint. RAM use is bounded by roughly one input CAF plus small
# temporary NumPy arrays.

def _empty_state():
    return {
        "next_file": 0,
        "n_hits": 0,
        "n_valid_output": 0,
        "n_truth_matched": 0,
        "n_truth_isnu_matched": 0,
        "n_broken_chain": 0,
        "n_invisible": 0,
        "n_noise": 0,
        "n_failed": 0,
        "true_counts": np.zeros(N_CLASSES, dtype=np.int64),
        "pred_counts": np.zeros(N_CLASSES, dtype=np.int64),
        "cm": np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64),
        "xfilter_counts": np.zeros(len(X_FILTER_BINS) - 1, dtype=np.int64),
        "sim_frac_per_event": np.empty(0, dtype=np.float32),
        "event_cm_array": np.empty((0, N_CLASSES, N_CLASSES), dtype=np.int32),
        "failed_indices": np.empty(0, dtype=np.int32),
    }


def _save_state(state):
    tmp = AGGREGATE_CHECKPOINT + ".tmp.npz"
    np.savez_compressed(tmp, **state)
    os.replace(tmp, AGGREGATE_CHECKPOINT)


def _load_state():
    if START_FRESH and os.path.exists(AGGREGATE_CHECKPOINT):
        os.remove(AGGREGATE_CHECKPOINT)

    if not os.path.exists(AGGREGATE_CHECKPOINT):
        return _empty_state()

    with np.load(AGGREGATE_CHECKPOINT, allow_pickle=False) as saved:
        state = {key: saved[key] for key in saved.files}

    # Scalars are restored as 0-D NumPy arrays; convert them to Python ints.
    for key in [
        "next_file", "n_hits", "n_valid_output", "n_truth_matched",
        "n_truth_isnu_matched", "n_broken_chain", "n_invisible",
        "n_noise", "n_failed",
    ]:
        state[key] = int(state[key])

    return state


def _process_one_file(path, file_index):
    # NumPy output avoids building a pandas DataFrame for every file.
    with uproot.open(path) as root_file:
        arrays = root_file[tree_path].arrays(load_cols, library="np")

    event = np.asarray(arrays["event"])
    truth_sem = np.asarray(arrays["truth_semantic"])
    truth_isnu = np.asarray(arrays["truth_isnu"])
    truth_g4id = np.asarray(arrays["truth_g4id"])
    has_output = np.asarray(arrays["has_ng_output"]) == 1

    n_rows = len(event)
    matched = truth_g4id >= 0

    result = {
        "n_hits": n_rows,
        "n_valid_output": int(has_output.sum()),
        "n_truth_matched": int(matched.sum()),
        "n_truth_isnu_matched": int((matched & (truth_isnu == 1)).sum()),
        "n_broken_chain": int((truth_sem == -2).sum()),
        "n_invisible": int((truth_sem == 5).sum()),
        "n_noise": int((truth_sem == -1).sum()),
        "true_counts": np.array(
            [(truth_sem == c).sum() for c in range(N_CLASSES)],
            dtype=np.int64,
        ),
        "pred_counts": np.zeros(N_CLASSES, dtype=np.int64),
        "cm": np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64),
        "xfilter_counts": np.zeros(len(X_FILTER_BINS) - 1, dtype=np.int64),
    }

    # Compact per-event information.
    unique_events, event_codes = np.unique(event, return_inverse=True)
    n_file_events = len(unique_events)

    event_hit_counts = np.bincount(event_codes, minlength=n_file_events)
    event_sim_counts = np.bincount(
        event_codes,
        weights=matched.astype(np.int8),
        minlength=n_file_events,
    )
    result["sim_frac_per_event"] = (
        event_sim_counts / event_hit_counts
    ).astype(np.float32)

    event_cms = np.zeros(
        (n_file_events, N_CLASSES, N_CLASSES), dtype=np.int32
    )

    if has_output.any():
        scores = np.column_stack(
            [np.asarray(arrays[col])[has_output] for col in score_cols]
        )
        assigned = scores.argmax(axis=1)
        result["pred_counts"] += np.bincount(
            assigned, minlength=N_CLASSES
        ).astype(np.int64)
        del scores, assigned

    no_truth_valid = (~matched) & has_output
    if no_truth_valid.any():
        hist, _ = np.histogram(
            np.asarray(arrays["x_filter"])[no_truth_valid],
            bins=X_FILTER_BINS,
        )
        result["xfilter_counts"] += hist

    valid_cm = has_output & (truth_sem >= 0) & (truth_sem < N_CLASSES)
    if valid_cm.any():
        valid_indices = np.flatnonzero(valid_cm)
        scores_cm = np.column_stack(
            [np.asarray(arrays[col])[valid_indices] for col in score_cols]
        )
        assigned_cm = scores_cm.argmax(axis=1)
        truth_cm = truth_sem[valid_indices].astype(np.int64, copy=False)
        event_cm_codes = event_codes[valid_indices]

        flat_pair = truth_cm * N_CLASSES + assigned_cm
        result["cm"] += np.bincount(
            flat_pair, minlength=N_CLASSES * N_CLASSES
        ).reshape(N_CLASSES, N_CLASSES)

        np.add.at(
            event_cms.reshape(n_file_events, -1),
            (event_cm_codes, flat_pair),
            1,
        )

        del valid_indices, scores_cm, assigned_cm
        del truth_cm, event_cm_codes, flat_pair

    result["event_cm_array"] = event_cms
    return result


state = _load_state()
start_index = state["next_file"]

print(f"resuming at file {start_index:,} / {len(all_paths):,}")
print(
    f"checkpoint already contains {state['n_hits']:,} hits and "
    f"{len(state['sim_frac_per_event']):,} events"
)

# Accumulate new event arrays in small lists and merge only at checkpoint time.
new_sim_frac = []
new_event_cms = []
new_failed = []
t0 = time.time()

for i in range(start_index, len(all_paths)):
    try:
        result = _process_one_file(all_paths[i], i)

        for key in [
            "n_hits", "n_valid_output", "n_truth_matched",
            "n_truth_isnu_matched", "n_broken_chain",
            "n_invisible", "n_noise",
        ]:
            state[key] += result[key]

        for key in ["true_counts", "pred_counts", "cm", "xfilter_counts"]:
            state[key] += result[key]

        new_sim_frac.append(result["sim_frac_per_event"])
        new_event_cms.append(result["event_cm_array"])

    except Exception as exc:
        state["n_failed"] += 1
        new_failed.append(i)
        print(f"\nFAILED file {i}: {all_paths[i]}")
        print(f"  {type(exc).__name__}: {exc}")

    state["next_file"] = i + 1

    should_checkpoint = (
        (i + 1) % CHECKPOINT_EVERY == 0
        or i + 1 == len(all_paths)
    )

    if should_checkpoint:
        if new_sim_frac:
            state["sim_frac_per_event"] = np.concatenate(
                [state["sim_frac_per_event"], *new_sim_frac]
            )
            state["event_cm_array"] = np.concatenate(
                [state["event_cm_array"], *new_event_cms], axis=0
            )
            new_sim_frac.clear()
            new_event_cms.clear()

        if new_failed:
            state["failed_indices"] = np.concatenate(
                [
                    state["failed_indices"],
                    np.asarray(new_failed, dtype=np.int32),
                ]
            )
            new_failed.clear()

        _save_state(state)
        elapsed = time.time() - t0
        rate = (i + 1 - start_index) / elapsed if elapsed else float("nan")
        remaining = len(all_paths) - (i + 1)
        eta_min = remaining / rate / 60 if rate > 0 else float("nan")

        print(
            f"checkpoint: {i+1:,}/{len(all_paths):,} files | "
            f"{state['n_hits']:,} hits | "
            f"{len(state['sim_frac_per_event']):,} events | "
            f"{state['n_failed']} failed | "
            f"{rate:.2f} files/s | ETA {eta_min:.1f} min"
        )
        gc.collect()

print("streaming pass complete")

In [ ]:
# Load the compact aggregate checkpoint for plotting.
#
# With MAX_FILES=200, this loads the 200-file test result and allows every
# downstream table and plot to run. With MAX_FILES=None, it expects the full run.
# The checkpoint itself remains small enough for an ordinary Jupyter session.

with np.load(AGGREGATE_CHECKPOINT, allow_pickle=False) as saved:
    aggregate = {key: saved[key] for key in saved.files}

if int(aggregate["next_file"]) != len(all_paths):
    raise RuntimeError(
        f"streaming pass is incomplete: processed "
        f"{int(aggregate['next_file'])}/{len(all_paths)} files"
    )

n_hits = int(aggregate["n_hits"])
n_valid_output = int(aggregate["n_valid_output"])
n_truth_matched = int(aggregate["n_truth_matched"])
n_truth_isnu_matched = int(aggregate["n_truth_isnu_matched"])
n_broken_chain = int(aggregate["n_broken_chain"])
n_invisible = int(aggregate["n_invisible"])
n_noise = int(aggregate["n_noise"])
n_failed = int(aggregate["n_failed"])

true_counts = aggregate["true_counts"]
pred_counts = aggregate["pred_counts"]
cm = aggregate["cm"]
xfilter_counts = aggregate["xfilter_counts"]
sim_frac_per_event = aggregate["sim_frac_per_event"]
event_cm_array = aggregate["event_cm_array"]
failed_indices = aggregate["failed_indices"]
n_events = len(sim_frac_per_event)

print(f"loaded compact aggregates for {n_hits:,} hits from {n_events:,} events")
print(f"failed input files: {n_failed}")
print(
    f"event-level bootstrap array: "
    f"{event_cm_array.nbytes / 1024**2:.1f} MiB"
)

## Check 1 -- truth-match stats

For an overlay sample, expect the truth-match rate to be much LOWER than
a pure-MC sample (MPVMPR was 97.7%) -- most hits here are real cosmic data
with no simulation truth at all, so a low match rate is the correct,
expected result, not a regression. What matters is that among the hits
that DO match, they should track back to the neutrino interaction
(`truth_isnu==1`) at a high rate, and the two failure-mode sentinels
(`-2`, `5`) should stay small.

In [ ]:
has_output_frac = n_valid_output / n_hits
truth_matched_frac = n_truth_matched / n_hits
nu_frac_of_matched = (
    n_truth_isnu_matched / n_truth_matched
    if n_truth_matched else float("nan")
)
broken_chain_frac = n_broken_chain / n_hits
invisible_frac = n_invisible / n_hits

summary = pd.DataFrame({
    "check": [
        "hits with valid NG output",
        "hits with a truth match (truth_g4id >= 0) -- expect LOW for overlay",
        "of MATCHED hits, fraction from the neutrino interaction -- expect HIGH",
        "broken particle-chain hits (truth_semantic == -2) -- expect ~0",
        "invisible-particle hits (truth_semantic == 5) -- expect small",
    ],
    "fraction": [
        has_output_frac,
        truth_matched_frac,
        nu_frac_of_matched,
        broken_chain_frac,
        invisible_frac,
    ],
    "n_hits": [
        n_valid_output,
        n_truth_matched,
        n_truth_isnu_matched,
        n_broken_chain,
        n_invisible,
    ],
})
summary["fraction"] = summary["fraction"].map(lambda x: f"{x:.2%}")
summary

## Check 1b -- overlay (simulated) content vs. real data -- and a real limitation of truth alone

`truth_g4id >= 0` (equivalently `truth_semantic != -1`) does correctly
separate "matched to the simulated overlay's neutrino interaction" from
"no simulated truth found." That part is solid -- confirmed below.

**But `-1` is NOT the same as "cosmic ray."** Overlay's non-simulated
component is real detector data, and real data carries no G4 truth
*regardless of whether the hit is a genuine cosmic-ray energy deposit or
pure electronic/detector noise* -- both look identical from the truth
side, since there's no simulation to consult either way. An earlier
version of this notebook mislabeled the whole `-1` population "real
cosmic," which overclaims what truth-matching can actually tell you.

**What can separate real activity from pure noise inside that `-1`
bucket**: `x_filter`, NuGraph's own filter-head score -- trained
specifically on "does this hit have a genuine energy deposit at all"
(confirmed against `hitgraph.py`'s `filter_label` convention, back when
building the semantic-label port), independent of simulated truth
entirely. This is the right tool for "is this real activity or noise,"
not truth matching, which structurally can't answer that question for
non-simulated data.

**Caveat**: `x_filter`'s exact convention (does higher mean "more likely
real" or "more likely noise") is inferred from the training-code label
definition, not independently re-confirmed against your specific
production's filter head -- worth sanity-checking the plot below against
that assumption rather than trusting the direction blindly.

In [ ]:
print(
    f"Overall: {truth_matched_frac:.1%} of all hits matched simulated "
    f"(overlay) truth, {1-truth_matched_frac:.1%} did not "
    f"(real data -- cosmic energy AND/OR pure noise, not distinguishable "
    f"by truth alone)"
)
print()
print("Per-event simulated-truth-hit-fraction distribution:")
print(pd.Series(sim_frac_per_event).describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(
    sim_frac_per_event,
    bins=30,
    color=MANAGUA(0.3),
    edgecolor="white",
    linewidth=0.5,
)
axes[0].set_xlabel("fraction of an event's hits matched to simulated truth")
axes[0].set_ylabel("events")
axes[0].set_title("Simulated-truth content per event")

xfilter_centres = 0.5 * (X_FILTER_BINS[:-1] + X_FILTER_BINS[1:])
axes[1].bar(
    xfilter_centres,
    xfilter_counts,
    width=np.diff(X_FILTER_BINS),
    color=MANAGUA(0.75),
    edgecolor="white",
    linewidth=0.4,
)
axes[1].set_xlabel("x_filter (NuGraph filter-head score)")
axes[1].set_ylabel("hits")
axes[1].set_title(
    "x_filter within the no-truth-match ('-1') population\n"
    "(higher = more likely real activity, per hitgraph.py convention)"
)

plt.tight_layout()
plt.savefig("bnb_overlay_content_per_event.png", dpi=150)
plt.show()

## Check 2 -- semantic label distributions

True label (this run), NuGraph's predicted label (argmax), and Giulia's
training-truth reference (thesis Fig. 4.10, MPVMPR sample -- shape
reference only, not a target, given the different event topology).

**Noise (`truth_semantic == -1`) is excluded from this plot** -- for an
overlay sample it dominates (mostly real cosmic data with no truth to
match), and including it would squash the 5 real classes down to
barely-visible slivers. The noise fraction itself is already reported in
Check 1 above, not lost, just not re-plotted here. True/Giulia bars are
each renormalized to sum to 1 over the 5 real classes only, so they're
directly comparable to the predicted-label bars (which are inherently a
5-way split already, argmax has no "noise" option).

In [ ]:
plot_labels = {0: "MIP", 1: "HIP", 2: "Shower", 3: "Michel", 4: "Diffuse"}
values = sorted(plot_labels.keys())

true_freq = true_counts / true_counts.sum()
pred_freq = pred_counts / pred_counts.sum()

# Giulia's reference (Fig. 4.10, read off by eye -- MPVMPR sample, shape ref. only),
# renormalized the same way (drop noise, renormalize over the 5 real classes)
# giulia_raw = np.array([0.297, 0.190, 0.430, 0.013, 0.035])
# giulia_freq = giulia_raw / giulia_raw.sum()

noise_frac = n_noise / n_hits
print(
    f"(noise fraction excluded from this plot: {noise_frac:.1%} "
    f"of all hits -- see Check 1)"
)

x = np.arange(len(values))
width = 0.27
colors = managua_discrete(3)

fig, ax = plt.subplots(figsize=(10, 6), dpi=350)
ax.bar(
    x - width, true_freq, width,
    label="True label (this run, noise excluded)", color=colors[0]
)
ax.bar(
    x, pred_freq, width,
    label="Predicted label (argmax, this run)", color=colors[1]
)
#ax.bar(
#    x + width, giulia_freq, width,
#    label="Giulia's MPVMPR training truth "
#          "(Fig. 4.10, approx., shape ref. only)",
#    color=colors[2],
#)

ax.set_xticks(x)
ax.set_xticklabels([plot_labels[v] for v in values])
ax.set_xlabel("semantic label")
ax.set_ylabel("frequency (fraction of the 5 real classes)")
ax.set_title(
    "Semantic label distribution: true vs. predicted vs. reference",
    fontsize=13,
)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig("bnb_overlay_semantic_distribution.png", dpi=350)
plt.show()

## Check 3 -- confusion matrix

Filters to hits that are (a) valid NG predictions and (b) one of the 5
trained classes (`truth_semantic` in `[0,4]`) -- this excludes the ~88%+
of hits that are real cosmic data with no truth match at all (`-1`), plus
`-2`/`5` if present. What's left is, functionally, "hits matched to real
simulated truth" -- almost entirely the overlaid neutrino interaction.

In [ ]:
n_cm_hits = int(cm.sum())
print(
    f"hits used for confusion matrix: {n_cm_hits:,} / "
    f"{n_hits:,} ({n_cm_hits/n_hits:.1%})"
)

with np.errstate(invalid="ignore", divide="ignore"):
    recall = cm / cm.sum(axis=1, keepdims=True)
    precision = cm / cm.sum(axis=0, keepdims=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5),dpi=350)

for ax, mat, title in zip(axes, [recall, precision], ["Semantic Recall", "Semantic Precision"]):
    im = ax.imshow(mat, cmap=MANAGUA, vmin=0, vmax=1, origin="lower")  # MIP at bottom, not top
    for i in range(5):
        for j in range(5):
            val = mat[i, j]
            txt_color = "white" if (val > 0.6 or val < 0.15) else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=11)
    ax.set_xticks(range(5)); ax.set_xticklabels(class_labels)
    ax.set_yticks(range(5)); ax.set_yticklabels(class_labels)
    ax.set_xlabel("Assigned label"); ax.set_ylabel("True label")
    ax.set_title(title, fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="fraction")

plt.tight_layout()
plt.savefig("bnb_overlay_confusion_matrix.png", dpi=350)
plt.show()

In [ ]:
per_class = pd.DataFrame({
    "class": class_labels,
    "n_true": cm.sum(axis=1),
    "n_predicted": cm.sum(axis=0),
    "recall (diag)": np.diag(recall).round(3),
    "precision (diag)": np.diag(precision).round(3),
})
per_class

## Statistical rigor: event-level bootstrap

Hits within one event are correlated (same tracks/showers), not
independent samples -- resampling individual hits would understate true
uncertainty. This resamples whole EVENTS with replacement instead,
recomputing the confusion matrix per resample.

Precomputes each event's confusion-matrix contribution once, then sums
small 5x5 matrices per draw -- this matters once you're past a handful of
events (the naive "filter the full table inside the bootstrap loop"
approach is fine at 5 events, effectively hangs at hundreds/thousands).

In [ ]:
# Event-level bootstrap using the compact (n_events, 5, 5) array built
# during streaming.  No hit-level dataframe or pandas groupby is needed.
#
# 500 replicas is a practical default for an interactive notebook. Increase
# N_BOOT to 1000 for a final result if desired.

N_BOOT = 500
BOOT_SEED = 0
rng = np.random.default_rng(BOOT_SEED)

event_cm_flat = event_cm_array.reshape(n_events, -1).astype(np.int64, copy=False)
boot_recall = np.full((N_BOOT, N_CLASSES), np.nan)
boot_precision = np.full((N_BOOT, N_CLASSES), np.nan)

for b in range(N_BOOT):
    sampled_indices = rng.integers(0, n_events, size=n_events)
    cm_b = event_cm_flat[sampled_indices].sum(axis=0).reshape(
        N_CLASSES, N_CLASSES
    )

    with np.errstate(invalid="ignore", divide="ignore"):
        boot_recall[b] = np.diag(cm_b) / cm_b.sum(axis=1)
        boot_precision[b] = np.diag(cm_b) / cm_b.sum(axis=0)

    if (b + 1) % 100 == 0:
        print(f"  bootstrap replicas: {b + 1}/{N_BOOT}")

recall_lo, recall_hi = np.nanpercentile(
    boot_recall, [16, 84], axis=0
)
prec_lo, prec_hi = np.nanpercentile(
    boot_precision, [16, 84], axis=0
)
recall_point = np.diag(recall)
prec_point = np.diag(precision)

boot_summary = pd.DataFrame({
    "class": class_labels,
    "recall": recall_point.round(3),
    "recall 68% CI (event bootstrap)": [
        f"[{lo:.2f}, {hi:.2f}]"
        for lo, hi in zip(recall_lo, recall_hi)
    ],
    "precision": prec_point.round(3),
    "precision 68% CI (event bootstrap)": [
        f"[{lo:.2f}, {hi:.2f}]"
        for lo, hi in zip(prec_lo, prec_hi)
    ],
})
boot_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=350)
colors = managua_discrete(5)

for ax, point, lo, hi, title in zip(
    axes, [recall_point, prec_point], [recall_lo, prec_lo], [recall_hi, prec_hi],
    ["Recall", "Precision"],
):
    yerr = np.vstack([point - lo, hi - point])
    yerr = np.clip(yerr, 0, None)
    ax.bar(class_labels, point, color=colors, yerr=yerr, capsize=4,
           error_kw={"elinewidth": 1.3, "ecolor": "black"})
    ax.set_ylim(0, 1)
    ax.set_ylabel(title.lower())
    ax.set_title(f"{title} with 68% event-level bootstrap CI ({n_events} events)", fontsize=12)

plt.tight_layout()
plt.savefig("bnb_overlay_bootstrap_uncertainty.png", dpi=350)
plt.show()

**Exact enumeration cross-check** (only meaningful/tractable for small
event counts, e.g. a handful of files) is intentionally not included here
-- past a dozen or so events, `C(2N-1,N)` possible resamples is
astronomically large. See `truth_analysis.ipynb` (the original MPVMPR
notebook) if you want to see that check on a small sample; for this BNB
overlay analysis at scale, the random 1000-draw bootstrap above is the
right tool.

## Is there an alternative to bootstrapping now that there are many more events?

**Short answer: the event-level bootstrap above is still the right tool,
and it's not slow anymore** -- it's already the fast, precomputed version
(`event_cms` built once, then each draw just sums small 5x5 matrices), so
"many more events" mostly means "more draws to sum over," not "the
approach doesn't scale."

**The reason to keep it, not switch to a plain per-hit interval:** hits
within one event are still correlated (same tracks/showers) no matter how
many events you have -- more events don't fix that, they just mean you
have more independent *clusters* to resample from, which is exactly what
the event-level bootstrap already exploits.

**A legitimate faster alternative, with a real caveat**: a plain per-hit
Wilson score interval (treating every hit as independent) is much cheaper
to compute and will converge to roughly the same point estimate. But it
**ignores clustering entirely** and will be *artificially narrow* --
understating the true uncertainty, sometimes substantially, especially for
rare classes where a handful of events can dominate the hit count. Useful
as a **fast sanity-check floor** (if the two disagree a lot, that itself
tells you something about how clustered your rare classes are), not as a
replacement.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# Fast, per-hit Wilson interval -- ignores event clustering, see caveat above.
# Compare its width against the event-bootstrap CI already computed: if the
# Wilson interval is much narrower, that's a sign of real clustering.
wilson_recall_lo, wilson_recall_hi = np.full(5, np.nan), np.full(5, np.nan)
wilson_prec_lo, wilson_prec_hi = np.full(5, np.nan), np.full(5, np.nan)

for c in range(5):
    n_true_c = cm[c, :].sum()
    if n_true_c > 0:
        wilson_recall_lo[c], wilson_recall_hi[c] = proportion_confint(cm[c, c], n_true_c, alpha=0.32, method="wilson")
    n_pred_c = cm[:, c].sum()
    if n_pred_c > 0:
        wilson_prec_lo[c], wilson_prec_hi[c] = proportion_confint(cm[c, c], n_pred_c, alpha=0.32, method="wilson")

compare_ci = pd.DataFrame({
    "class": class_labels,
    "recall": recall_point.round(3),
    "recall 68% CI (event bootstrap)": [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(recall_lo, recall_hi)],
    "recall 68% CI (naive Wilson, per-hit)": [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(wilson_recall_lo, wilson_recall_hi)],
    "precision": prec_point.round(3),
    "precision 68% CI (event bootstrap)": [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(prec_lo, prec_hi)],
    "precision 68% CI (naive Wilson, per-hit)": [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(wilson_prec_lo, wilson_prec_hi)],
})
compare_ci